# RAGScholar — Demo Final

**Purpose:** build one academic ML textbook from the original PDF into a clean four-modality retrieval system: **Text + Math + Visual + Table**.

This notebook is a **fresh build**. It does not reuse any document-specific cache, FAISS index, BM25 index, serialized runtime, or experimental checkpoint from the old notebook. Pretrained model weights may still come from the local Hugging Face cache.

The notebook keeps the final engineering decisions from the experimental work:

- source-faithful canonical knowledge objects
- Qwen3 dense retrieval + BM25 + RRF + Qwen3 reranking for text
- equation cards + source graph + role-aware math resolution
- source-caption/Qwen + multi-view SigLIP2 + auxiliary OCR lexical retrieval for visuals
- structured table/cell records with table- and row-level hybrid retrieval
- deterministic query planning, page/reference handling, safe abstention, and one unified `retrieve()` API

**Scope:** retrieval only. Answer generation and arbitrary numerical computation are intentionally outside this notebook.

# 0. Configuration

All paths, model IDs, retrieval parameters, and fresh-build behavior are defined here. Change only this section when moving the demo to another machine or source PDF.

In [9]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import hashlib, json, math, os, re, shutil, sys, unicodedata

import numpy as np
import pandas as pd
import torch
import pymupdf
from bs4 import BeautifulSoup
from PIL import Image
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() in {"notebook", "notebooks"} else cwd
SOURCE_PDF = PROJECT_ROOT / "data" / "Textbooks" / "UnderstandingDeepLearning_02_09_26_C.pdf"
ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts" / "books"

PARSE_TIER = "advanced"
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
RERANK_MODEL = "Qwen/Qwen3-Reranker-0.6B"
SIGLIP_MODEL = "google/siglip2-base-patch16-224"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESET_BUILD = True
ENABLE_AUX_OCR = False

TEXT_CFG = {
    "target_tokens": 350, "soft_tokens": 420, "max_tokens": 500, "min_tokens": 80,
    "dense_k": 30, "bm25_k": 30, "rrf_k": 60, "rrf_top_k": 30,
    "anchor_budget": 6, "neighbor_budget": 6,
}
MATH_CFG = {
    "dense_k": 60, "bm25_k": 60, "source_chunk_k": 30, "source_equation_k": 40,
    "rrf_k": 60, "rerank_pool": 100,
}
VISUAL_CFG = {
    "candidate_k": 30, "evidence_k": 3, "rrf_k": 60, "rerank_pool": 12,
    "semantic_weights": {"qwen_source": 1.0, "bm25_source": 1.0, "ocr_lexical": 0.25},
    "visual_weights": {"qwen_source": 1.0, "siglip_multiview": 1.0, "ocr_lexical": 0.5, "bm25_source": 0.25},
}
TABLE_CFG = {
    "table_k": 20, "row_k": 30, "rrf_k": 60, "evidence_k": 5,
    "min_rows": 2, "min_cols": 2, "min_nonempty": 4, "min_density": 0.30,
}

print("Project:", PROJECT_ROOT)
print("Source :", SOURCE_PDF)
print("Device :", DEVICE)

Project: C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt
Source : C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Textbooks\UnderstandingDeepLearning_02_09_26_C.pdf
Device : cuda


## 0.1 Utilities

Small shared helpers for hashing, JSON/JSONL persistence, text normalization, technical tokenization, and rank fusion.

In [10]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(chunk_size), b""):
            h.update(part)
    return h.hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

def clean_text(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()

def short_text(value, n=220):
    text = clean_text(value)
    return text[:n] + ("..." if len(text) > n else "")

def stable_hash(*parts, n=12):
    payload = json.dumps(parts, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()[:n]

TECH_TOKEN_RE = re.compile(r"[^\W\s]+(?:[./×+\-][^\W\s]+)*", re.UNICODE)
TECH_SPLIT_RE = re.compile(r"[./×+\-]")

def technical_split(text):
    out = []
    for token in TECH_TOKEN_RE.findall(str(text).lower()):
        out.append(token)
        if TECH_SPLIT_RE.search(token):
            out.extend(p for p in TECH_SPLIT_RE.split(token) if p and (len(p) >= 2 or p.isdigit()))
    return out

def weighted_rrf(lanes, weights, k=60, top_k=30, id_key="id"):
    scores, provenance = defaultdict(float), defaultdict(dict)
    for lane, rows in lanes.items():
        weight = float(weights.get(lane, 1.0))
        for row in rows:
            rid, rank = row[id_key], int(row["rank"])
            contribution = weight / (k + rank)
            scores[rid] += contribution
            provenance[rid][lane] = {"rank": rank, "score": float(row.get("score", 0.0)), "weight": weight}
    ranked = sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:top_k]
    return [{id_key: rid, "rank": i, "score": float(score), "lanes": provenance[rid]} for i, (rid, score) in enumerate(ranked, 1)]

def package_version(name):
    try: return version(name)
    except PackageNotFoundError: return None

# 1. Source Ingestion

The original PDF is the authority for document identity, physical pages, printed labels, geometry, and source crops. A fresh artifact directory is created from the PDF hash.

In [11]:
if not SOURCE_PDF.is_file():
    raise FileNotFoundError(SOURCE_PDF)

BOOK_SHA256 = sha256_file(SOURCE_PDF)
DOCUMENT_ID = f"book_{BOOK_SHA256[:12]}"
BOOK_DIR = ARTIFACTS_ROOT / DOCUMENT_ID

if RESET_BUILD and BOOK_DIR.exists():
    shutil.rmtree(BOOK_DIR)

DIRS = {
    "parse": BOOK_DIR / "parse",
    "canonical": BOOK_DIR / "canonical",
    "text": BOOK_DIR / "text",
    "math": BOOK_DIR / "math",
    "visual": BOOK_DIR / "visual",
    "table": BOOK_DIR / "table",
    "validation": BOOK_DIR / "validation",
}
for path in DIRS.values(): path.mkdir(parents=True, exist_ok=True)

pdf = pymupdf.open(SOURCE_PDF)
PAGES = [{
    "page_index": i,
    "pdf_page_number": i + 1,
    "printed_page_label": pdf[i].get_label() or None,
    "width_pt": float(pdf[i].rect.width),
    "height_pt": float(pdf[i].rect.height),
    "rotation": int(pdf[i].rotation),
} for i in range(pdf.page_count)]

TOC = [{"level": int(level), "title": title.strip(), "page_index": max(int(page_no) - 1, 0)}
       for level, title, page_no in pdf.get_toc(simple=True)]

SOURCE = {
    "document_id": DOCUMENT_ID,
    "filename": SOURCE_PDF.name,
    "source_path": str(SOURCE_PDF.resolve()),
    "sha256": BOOK_SHA256,
    "page_count": pdf.page_count,
    "pdf_metadata": pdf.metadata,
}
write_json(BOOK_DIR / "config.json", {
    "document_id": DOCUMENT_ID, "source": SOURCE,
    "parser": {"name": "mineru", "tier": PARSE_TIER, "ocr_mode": "auto"},
    "models": {"embedding": EMBED_MODEL, "reranker": RERANK_MODEL, "siglip": SIGLIP_MODEL},
    "text": TEXT_CFG, "math": MATH_CFG, "visual": VISUAL_CFG, "table": TABLE_CFG,
})

print("Document:", DOCUMENT_ID)
print("Pages   :", pdf.page_count)
print("Artifacts:", BOOK_DIR)

Document: book_f8237d393163
Pages   : 541
Artifacts: C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\artifacts\books\book_f8237d393163


# 2. Semantic Parsing

MinerU parses the entire PDF from scratch. Parser-specific payloads are preserved, while later sections consume a stable RAGScholar schema.

In [14]:
from concurrent.futures import ThreadPoolExecutor

from mineru.parser import parse
from mineru.parser.writer import FileBasedDataWriter
from mineru.render import render, RenderFormat

MINERU_DIR = DIRS["parse"] / "mineru"
MINERU_DIR.mkdir(parents=True, exist_ok=True)

def run_mineru():
    return parse(str(SOURCE_PDF), tier=PARSE_TIER, ocr_mode="auto", page_range="all")

with ThreadPoolExecutor(max_workers=1) as executor:
    parse_result = executor.submit(run_mineru).result()

parse_result.save(FileBasedDataWriter(str(MINERU_DIR)))

CONTENT_PAGES = render(parse_result.middle_json, RenderFormat.CONTENT_LIST_V2)
structured_content = parse_result.structured_content()

PARSER_META = {
    "name": "mineru",
    "tier_requested": PARSE_TIER,
    "producer": structured_content.get("metadata", {}).get("producer", {}),
    "extensions": structured_content.get("extensions", {}),
}

print("Parsed pages:", len(CONTENT_PAGES))

2026-10-05 13:22:38.620 | INFO     | mineru.model.vlm.selector:get_vlm_engine:61 - Using llama-cpp-engine as the inference engine for VLM.
2026-10-05 13:22:38.624 | INFO     | mineru.backend.analysis.pdf.window:_log_processing_window_plan:120 - Hybrid processing-window run. page_count=541, window_size=64, total_windows=9
2026-10-05 13:22:40.446 | INFO     | mineru.backend.analysis.pdf.window:_log_processing_window:128 - Hybrid processing window 1/9: pages 1-64/541 (64 pages)
OCR-rec Predict: 100%|██████████| 6/6 [00:00<00:00, 59.47it/s]
2026-10-05 13:24:54.569 | INFO     | mineru.backend.analysis.pdf.window:_log_processing_window:128 - Hybrid processing window 2/9: pages 65-128/541 (64 pages)
OCR-det: 100%|██████████| 642/642 [00:03<00:00, 193.30it/s]
2026-10-05 13:26:53.802 | INFO     | mineru.backend.analysis.pdf.window:_log_processing_window:128 - Hybrid processing window 3/9: pages 129-192/541 (64 pages)
OCR-rec Predict: 100%|██████████| 13/13 [00:00<00:00, 93.68it/s]
2026-10-05 13

Parsed pages: 541


## 2.1 Parser Adapters

These helpers convert MinerU blocks into consistent text, geometry, asset, and table-cell representations.

In [15]:
def spans_to_text(value):
    if value is None: return ""
    if isinstance(value, str): return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, str): parts.append(item)
            elif isinstance(item, dict):
                t = str(item.get("type", "")).lower()
                c = spans_to_text(item.get("content", item.get("text", "")))
                parts.append(f"${c}$" if c and "equation" in t else c)
        return clean_text("".join(parts))
    if isinstance(value, dict): return spans_to_text(value.get("content", value.get("text", "")))
    return clean_text(value)

def canonical_type(mineru_type, payload):
    t = str(mineru_type or "").lower()
    if "title" in t: return "heading"
    if "paragraph" in t: return "paragraph"
    if "equation" in t: return "equation"
    if t == "image" or t.endswith("_image"): return "figure"
    if "chart" in t: return "chart"
    if "table" in t: return "table"
    if "algorithm" in t: return "algorithm"
    if "code" in t: return "code"
    if "index" in t: return "index"
    if "list" in t: return "reference" if "ref" in str(payload.get("list_type", "")).lower() else "list"
    if "page_header" in t or t == "header": return "header"
    if "page_footer" in t or t == "footer": return "footer"
    if "page_number" in t: return "page_number"
    if "footnote" in t: return "footnote"
    if "aside" in t: return "aside"
    return t or "unknown"

def block_text(block):
    t, c = str(block.get("type", "")).lower(), block.get("content") or {}
    if "title" in t: return spans_to_text(c.get("title_content"))
    if "paragraph" in t: return spans_to_text(c.get("paragraph_content"))
    if "equation" in t: return clean_text(c.get("math_content"))
    if t == "image": return spans_to_text(c.get("image_caption"))
    if "chart" in t: return spans_to_text(c.get("chart_caption"))
    if "table" in t: return spans_to_text(c.get("table_caption"))
    if "algorithm" in t: return spans_to_text(c.get("algorithm_content"))
    if "code" in t: return spans_to_text(c.get("code_content"))
    if "list" in t or "index" in t:
        return "\n".join(spans_to_text(x.get("item_content", x)) for x in c.get("list_items", []))
    for key, value in c.items():
        if key.endswith("_content"): return spans_to_text(value)
    return ""

def bbox_1000_to_pdf(bbox, page):
    if not bbox: return None
    x0, y0, x1, y1 = map(float, bbox)
    return [x0 * page["width_pt"] / 1000, y0 * page["height_pt"] / 1000,
            x1 * page["width_pt"] / 1000, y1 * page["height_pt"] / 1000]

def parser_asset(payload):
    raw = ((payload.get("image_source") or {}).get("path") or "").strip()
    if not raw: return None
    raw = raw.lstrip("/\\")
    direct, fallback = MINERU_DIR / raw, MINERU_DIR / "images" / Path(raw).name
    return str(direct if direct.exists() else fallback)

def html_to_cells(html):
    if not html: return []
    table = BeautifulSoup(html, "lxml").find("table")
    if table is None: return []
    occupied, cells = set(), []
    for r, tr in enumerate(table.find_all("tr")):
        c = 0
        for cell in tr.find_all(["th", "td"], recursive=False):
            while (r, c) in occupied: c += 1
            rs, cs = int(cell.get("rowspan", 1) or 1), int(cell.get("colspan", 1) or 1)
            cells.append({"row": r, "col": c, "rowspan": rs, "colspan": cs,
                          "is_header": cell.name == "th", "text": clean_text(cell.get_text(" ", strip=True))})
            for rr in range(r, r + rs):
                for cc in range(c, c + cs): occupied.add((rr, cc))
            c += cs
    return cells

# 3. Canonical Knowledge Document

Every retriever is built from one canonical source registry. Original source content and geometry stay authoritative; enriched representations are only retrieval views.

In [16]:
def object_id(page_idx, order, kind, content, bbox):
    return f"{DOCUMENT_ID}:p{page_idx:04d}:{kind}:{order:04d}:{stable_hash(kind, content, bbox, n=8)}"

def caption_footnote(kind, payload):
    keys = {
        "figure": ("image_caption", "image_footnote"), "chart": ("chart_caption", "chart_footnote"),
        "table": ("table_caption", "table_footnote"), "code": ("code_caption", "code_footnote"),
        "algorithm": ("algorithm_caption", "algorithm_footnote"),
    }
    a, b = keys.get(kind, (None, None))
    return (spans_to_text(payload.get(a)), spans_to_text(payload.get(b))) if a else (None, None)

OBJECTS, section_titles, section_ids = [], [], []
for page_idx, blocks in enumerate(CONTENT_PAGES):
    page = PAGES[page_idx]
    for order, block in enumerate(blocks):
        mineru_type, payload = block.get("type"), block.get("content") or {}
        kind, text, bbox_norm = canonical_type(mineru_type, payload), block_text(block), block.get("bbox")
        oid = object_id(page_idx, order, kind, text, bbox_norm)

        if kind == "heading":
            level = max(int(payload.get("level", 0) or 1), 1)
            section_titles, section_ids = section_titles[:level - 1], section_ids[:level - 1]
            parent_heading_id = section_ids[-1] if section_ids else None
            section_path = section_titles + [text]
        else:
            parent_heading_id = section_ids[-1] if section_ids else None
            section_path = list(section_titles)

        caption, footnote = caption_footnote(kind, payload)
        html = payload.get("html") if kind == "table" else None
        OBJECTS.append({
            "object_id": oid, "document_id": DOCUMENT_ID, "object_type": kind, "mineru_type": mineru_type,
            "page_index": page_idx, "pdf_page_number": page_idx + 1, "printed_page_label": page["printed_page_label"],
            "reading_order": order, "section_path": section_path, "parent_heading_id": parent_heading_id,
            "bbox_norm_1000": bbox_norm, "bbox_pdf": bbox_1000_to_pdf(bbox_norm, page), "content": text,
            "inline_spans": payload.get("paragraph_content") if kind == "paragraph" else payload.get("title_content") if kind == "heading" else None,
            "latex": payload.get("math_content") if kind == "equation" else None,
            "caption": caption, "footnote": footnote, "table_html": html,
            "table_cells": html_to_cells(html) if html else [],
            "asset_path": parser_asset(payload) if kind in {"figure", "chart", "table", "equation"} else None,
            "source_payload": payload, "parser": "mineru", "source_sha256": BOOK_SHA256,
            "derived": False, "indexable": kind not in {"header", "footer", "page_number", "aside", "reference"},
        })
        if kind == "heading":
            section_titles.append(text); section_ids.append(oid)

OBJECT_BY_ID = {o["object_id"]: o for o in OBJECTS}
OBJECTS_BY_TYPE = defaultdict(list)
for obj in OBJECTS: OBJECTS_BY_TYPE[obj["object_type"]].append(obj)

print("Knowledge objects:", len(OBJECTS))
print(pd.Series([o["object_type"] for o in OBJECTS]).value_counts().sort_index().to_string())

Knowledge objects: 7275
algorithm         3
aside            84
chart            53
code              1
equation        602
figure          260
footer          538
footnote         25
header          506
heading         555
list             46
page_number     501
paragraph      3998
reference        99
table             4


## 3.1 Labels, References, and Structural Links

Deterministic previous/next relations and explicit Figure/Table/Equation labels provide the source graph used by later retrieval lanes.

In [17]:
FIG_LABEL = re.compile(r"\b(?:fig(?:ure)?\.?)\s*(\d+(?:\.\d+)*)", re.I)
TABLE_LABEL = re.compile(r"\btable\s*(\d+(?:\.\d+)*)", re.I)
EQ_REF = re.compile(r"\b(?:eq(?:uation)?\.?)\s*(\d+(?:\.\d+)*)", re.I)
EQ_TAG = re.compile(r"\\tag\{([^}]+)\}")

label_to_id = {}
for obj in OBJECTS:
    label = None
    if obj["object_type"] in {"figure", "chart"}:
        m = FIG_LABEL.search(obj.get("caption") or ""); label = f"figure:{m.group(1)}" if m else None
    elif obj["object_type"] == "table":
        m = TABLE_LABEL.search(obj.get("caption") or ""); label = f"table:{m.group(1)}" if m else None
    elif obj["object_type"] == "equation":
        m = EQ_TAG.search(obj.get("latex") or ""); label = f"equation:{m.group(1)}" if m else None
    obj["label"] = label
    if label: label_to_id[label.lower()] = obj["object_id"]

ordered = sorted(OBJECTS, key=lambda o: (o["page_index"], o["reading_order"]))
for i, obj in enumerate(ordered):
    obj["previous_id"] = ordered[i - 1]["object_id"] if i else None
    obj["next_id"] = ordered[i + 1]["object_id"] if i + 1 < len(ordered) else None
    refs, text = [], obj.get("content") or ""
    for prefix, rx in (("figure", FIG_LABEL), ("table", TABLE_LABEL), ("equation", EQ_REF)):
        for number in rx.findall(text):
            target = label_to_id.get(f"{prefix}:{number}".lower())
            if target and target != obj["object_id"]: refs.append(target)
    obj["references"] = list(dict.fromkeys(refs))

## 3.2 Persist Canonical Source Registry

The canonical document is the single source of truth for all four retrieval lanes.

In [18]:
page_objects = defaultdict(list)
for obj in OBJECTS: page_objects[obj["page_index"]].append(obj["object_id"])

FINAL_PAGES = [dict(page, object_ids=page_objects[page["page_index"]]) for page in PAGES]
KNOWLEDGE_DOCUMENT = {
    "schema": "ragscholar.knowledge_document", "schema_version": "1.0",
    "source": SOURCE, "parser": PARSER_META, "toc": TOC, "pages": FINAL_PAGES, "objects": OBJECTS,
}
write_json(DIRS["canonical"] / "document.json", KNOWLEDGE_DOCUMENT)
write_jsonl(DIRS["canonical"] / "objects.jsonl", OBJECTS)
write_json(DIRS["canonical"] / "labels.json", label_to_id)

print("Canonical document:", DIRS["canonical"] / "document.json")

Canonical document: C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\artifacts\books\book_f8237d393163\canonical\document.json


# 4. Text RAG

Final Text lane: **structure-aware chunks → Qwen3 dense + BM25 → RRF → Qwen3 reranker → source-bound evidence + local expansion**.

## 4.1 Structure-Aware Chunks

In [19]:
from transformers import AutoTokenizer

TEXT_TYPES = {"heading", "paragraph", "list", "algorithm", "code", "footnote", "definition", "theorem", "proof", "worked_example"}
TEXT_OBJECTS = sorted([o for o in OBJECTS if o.get("indexable", True) and o["object_type"] in TEXT_TYPES and clean_text(o.get("content"))],
                      key=lambda o: (o["page_index"], o["reading_order"]))
TEXT_OBJECT_IDS = {o["object_id"] for o in TEXT_OBJECTS}
BRIDGE_TYPES = {"equation", "figure", "chart", "table"}
CHUNK_EXCLUDED_SECTIONS = {"index", "bibliography", "references"}
text_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL, use_fast=True)

def token_count(text): return len(text_tokenizer.encode(text, add_special_tokens=False))
def excluded_section(path): return any(str(x).strip().lower() in CHUNK_EXCLUDED_SECTIONS for x in path or [])
def retrieval_text(section, body):
    path = " > ".join(section)
    return f"Section: {path}\n\n{body}" if path else body

def split_long_text(text, limit):
    if token_count(text) <= limit: return [text]
    sentences, parts, buf = re.split(r"(?<=[.!?])\s+(?=\S)", text), [], ""
    for sentence in sentences:
        candidate = f"{buf} {sentence}".strip()
        if token_count(candidate) <= limit: buf = candidate
        else:
            if buf: parts.append(buf)
            if token_count(sentence) <= limit: buf = sentence
            else:
                ids = text_tokenizer.encode(sentence, add_special_tokens=False)
                parts.extend(text_tokenizer.decode(ids[i:i + limit], skip_special_tokens=True).strip() for i in range(0, len(ids), limit))
                buf = ""
    if buf: parts.append(buf)
    return [x for x in parts if x]

heading_by_path = {tuple(o["section_path"]): o["object_id"] for o in TEXT_OBJECTS if o["object_type"] == "heading" and not excluded_section(o["section_path"])}
SECTION_ITEMS = defaultdict(list)
for obj in ordered:
    section = obj.get("section_path") or []
    if excluded_section(section) or obj["object_type"] == "heading": continue
    if obj["object_id"] in TEXT_OBJECT_IDS:
        prefix = token_count(retrieval_text(section, ""))
        limit = max(TEXT_CFG["min_tokens"], TEXT_CFG["max_tokens"] - prefix)
        for part_no, part in enumerate(split_long_text(obj["content"], limit)):
            SECTION_ITEMS[tuple(section)].append({"kind": "text", "object_id": obj["object_id"], "part_no": part_no,
                                                  "text": part, "page_index": obj["page_index"], "references": obj.get("references", [])})
    elif obj["object_type"] in BRIDGE_TYPES:
        label = (obj.get("label") or obj["object_type"]).replace(":", " ").title()
        SECTION_ITEMS[tuple(section)].append({"kind": "bridge", "object_id": obj["object_id"], "text": f"[{label}]",
                                              "page_index": obj["page_index"], "references": []})

def body(items, include_bridges=True):
    return "\n\n".join(x["text"] for x in items if x["kind"] == "text" or include_bridges)

def make_chunk(section, items):
    text_items = [x for x in items if x["kind"] == "text"]
    source_ids = list(dict.fromkeys(x["object_id"] for x in text_items))
    bridge_ids = list(dict.fromkeys(x["object_id"] for x in items if x["kind"] == "bridge"))
    refs = list(dict.fromkeys(r for x in text_items for r in x["references"]))
    headings = [heading_by_path[tuple(section[:i])] for i in range(1, len(section) + 1) if tuple(section[:i]) in heading_by_path]
    source_text, search_text = body(items, False), retrieval_text(section, body(items, True))
    return {
        "chunk_id": f"{DOCUMENT_ID}:text:{stable_hash(source_ids, bridge_ids, source_text, n=10)}",
        "document_id": DOCUMENT_ID, "source_object_ids": source_ids, "bridge_object_ids": bridge_ids,
        "context_heading_ids": headings, "text": source_text, "retrieval_text": search_text,
        "section_path": list(section), "page_start": min(x["page_index"] for x in items), "page_end": max(x["page_index"] for x in items),
        "references": refs, "token_count": token_count(search_text),
    }

TEXT_CHUNKS = []
for section, items in SECTION_ITEMS.items():
    current, groups = [], []
    for item in items:
        candidate = current + [item]
        candidate_tokens = token_count(retrieval_text(section, body(candidate)))
        current_tokens = token_count(retrieval_text(section, body(current))) if current else 0
        if current and any(x["kind"] == "text" for x in current) and (candidate_tokens > TEXT_CFG["max_tokens"] or (current_tokens >= TEXT_CFG["target_tokens"] and candidate_tokens > TEXT_CFG["soft_tokens"])):
            groups.append(current); current = [item]
        else: current = candidate
    if current and any(x["kind"] == "text" for x in current): groups.append(current)
    if len(groups) >= 2:
        last_tokens = token_count(retrieval_text(section, body(groups[-1])))
        merged_tokens = token_count(retrieval_text(section, body(groups[-2] + groups[-1])))
        if last_tokens < TEXT_CFG["min_tokens"] and merged_tokens <= TEXT_CFG["max_tokens"]:
            groups[-2] += groups[-1]; groups.pop()
    TEXT_CHUNKS.extend(make_chunk(section, group) for group in groups)

TEXT_CHUNK_BY_ID = {c["chunk_id"]: c for c in TEXT_CHUNKS}
write_jsonl(DIRS["text"] / "chunks.jsonl", TEXT_CHUNKS)
print("Text chunks:", len(TEXT_CHUNKS))

Text chunks: 756


## 4.2 Dense, BM25, and Reranker Indexes

In [20]:
import faiss, bm25s
from bm25s.tokenization import Tokenizer
from sentence_transformers import SentenceTransformer, CrossEncoder

BODY_PAGES = [p["page_index"] for p in PAGES if re.fullmatch(r"\d+", str(p.get("printed_page_label") or "").strip())]
FIRST_BODY_PAGE = min(BODY_PAGES) if BODY_PAGES else 0
RETRIEVAL_EXCLUDED = {"index", "bibliography", "references", "contents", "table of contents", "attribution"}

def valid_text_chunk(chunk):
    section = [str(x).strip().lower() for x in chunk.get("section_path", [])]
    return chunk["page_start"] >= FIRST_BODY_PAGE and not any(x in RETRIEVAL_EXCLUDED for x in section)

def content_role(section):
    names = [str(x).strip().lower() for x in section]
    if "notes" in names: return "notes"
    if any(x.startswith("appendix") for x in names): return "appendix"
    if any(x in {"exercise", "exercises", "problem", "problems"} for x in names): return "exercise"
    return "main"

TEXT_RECORDS = []
for row_id, chunk in enumerate(c for c in TEXT_CHUNKS if valid_text_chunk(c)):
    section = chunk.get("section_path") or []
    bridge_terms = " ".join((OBJECT_BY_ID[oid].get("label") or OBJECT_BY_ID[oid]["object_type"]).replace(":", " ") for oid in chunk.get("bridge_object_ids", []) if oid in OBJECT_BY_ID)
    bm25_text = "\n".join(x for x in ((section[-1] if section else ""), chunk["text"], bridge_terms) if x)
    TEXT_RECORDS.append({
        "row_id": row_id, "chunk_id": chunk["chunk_id"], "document_id": DOCUMENT_ID,
        "source_text": chunk["text"], "dense_text": chunk["retrieval_text"], "bm25_text": bm25_text,
        "section_path": section, "content_role": content_role(section), "page_start": chunk["page_start"], "page_end": chunk["page_end"],
        "source_object_ids": chunk["source_object_ids"], "bridge_object_ids": chunk["bridge_object_ids"], "references": chunk["references"],
    })
TEXT_BY_ROW = {r["row_id"]: r for r in TEXT_RECORDS}
TEXT_BY_CHUNK = {r["chunk_id"]: r for r in TEXT_RECORDS}
write_jsonl(DIRS["text"] / "records.jsonl", TEXT_RECORDS)

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
encode_docs = embedder.encode_document if hasattr(embedder, "encode_document") else embedder.encode
text_vectors = encode_docs([r["dense_text"] for r in TEXT_RECORDS], batch_size=8, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=True).astype("float32")
text_vectors = np.ascontiguousarray(text_vectors); faiss.normalize_L2(text_vectors)
TEXT_DENSE_INDEX = faiss.IndexFlatIP(text_vectors.shape[1]); TEXT_DENSE_INDEX.add(text_vectors)
faiss.write_index(TEXT_DENSE_INDEX, str(DIRS["text"] / "dense.faiss"))

text_bm25_tokenizer = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None)
text_tokens = text_bm25_tokenizer.tokenize([r["bm25_text"] for r in TEXT_RECORDS], return_as="tuple")
TEXT_BM25 = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); TEXT_BM25.index(text_tokens)
TEXT_BM25_DIR = DIRS["text"] / "bm25"; TEXT_BM25_DIR.mkdir(parents=True, exist_ok=True)
TEXT_BM25.save(str(TEXT_BM25_DIR)); text_bm25_tokenizer.save_vocab(save_dir=str(TEXT_BM25_DIR)); text_bm25_tokenizer.save_stopwords(save_dir=str(TEXT_BM25_DIR))

TEXT_RERANK_PROMPT = "Rank textbook passages by how directly and accurately they provide evidence for the research question. Prefer explicit evidence over merely related passages."
MATH_RERANK_PROMPT = "Rank textbook equations by whether the equation and its local source context satisfy the exact mathematical role requested. Distinguish definitions from related formulas, probability from loss, forward from inverse, and Jacobian matrix from determinant."
RERANK_DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float16 if DEVICE == "cuda" else torch.float32
reranker = CrossEncoder(RERANK_MODEL, device=DEVICE, model_kwargs={"torch_dtype": RERANK_DTYPE},
                        prompts={"research": TEXT_RERANK_PROMPT, "math_equation": MATH_RERANK_PROMPT}, default_prompt_name="research")

print("Text records:", len(TEXT_RECORDS))

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/91 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/723 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/723 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Default prompt name is set to 'research'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


Text records: 723


## 4.3 Text Retrieval API

In [21]:
def encode_query(query):
    fn = embedder.encode_query if hasattr(embedder, "encode_query") else embedder.encode
    kwargs = {} if hasattr(embedder, "encode_query") else {"prompt_name": "query"}
    vec = fn([query], normalize_embeddings=True, convert_to_numpy=True, **kwargs).astype("float32")
    vec = np.ascontiguousarray(vec); faiss.normalize_L2(vec)
    return vec

def text_dense_search(query, k=None):
    k = min(k or TEXT_CFG["dense_k"], len(TEXT_RECORDS))
    scores, rows = TEXT_DENSE_INDEX.search(encode_query(query), k)
    return [{"id": TEXT_BY_ROW[int(row)]["chunk_id"], "rank": i, "score": float(score), "record": TEXT_BY_ROW[int(row)]}
            for i, (row, score) in enumerate(zip(rows[0], scores[0]), 1) if row >= 0]

def text_bm25_search(query, k=None):
    k = min(k or TEXT_CFG["bm25_k"], len(TEXT_RECORDS))
    tokens = text_bm25_tokenizer.tokenize([query], update_vocab=False, return_as="tuple")
    rows, scores = TEXT_BM25.retrieve(tokens, k=k, return_as="tuple", show_progress=False)
    hits = [{"id": TEXT_BY_ROW[int(row)]["chunk_id"], "score": float(score), "record": TEXT_BY_ROW[int(row)]}
            for row, score in zip(rows[0], scores[0]) if row >= 0 and float(score) > 0]
    for i, hit in enumerate(hits, 1): hit["rank"] = i
    return hits

def text_rrf(query):
    dense, lexical = text_dense_search(query), text_bm25_search(query)
    fused = weighted_rrf({"dense": dense, "bm25": lexical}, {"dense": 1.0, "bm25": 1.0}, TEXT_CFG["rrf_k"], TEXT_CFG["rrf_top_k"])
    pool = {x["id"]: x for x in dense + lexical}
    return [dict(item, record=pool[item["id"]]["record"]) for item in fused]

def rerank_text(query, candidates):
    if not candidates: return []
    scores = np.asarray(reranker.predict([(query, c["record"]["dense_text"]) for c in candidates], prompt_name="research",
                                         batch_size=4, show_progress_bar=False, convert_to_numpy=True)).reshape(-1)
    ranked = [dict(c, rerank_score=float(score)) for c, score in zip(candidates, scores)]
    ranked.sort(key=lambda x: (-x["rerank_score"], x["rank"]))
    return ranked

section_records = defaultdict(list)
for r in TEXT_RECORDS: section_records[tuple(r["section_path"])].append(r)
section_position = {r["chunk_id"]: (section, i) for section, records in section_records.items() for i, r in enumerate(records)}

def retrieve_text(query, top_k=6):
    ranked = rerank_text(query, text_rrf(query))
    anchors = ranked[:min(TEXT_CFG["anchor_budget"], len(ranked))]
    neighbors = {}
    anchor_ids = {a["record"]["chunk_id"] for a in anchors}
    for anchor in anchors:
        section, i = section_position[anchor["record"]["chunk_id"]]
        records = section_records[section]
        for j in (i - 1, i + 1):
            if 0 <= j < len(records) and records[j]["chunk_id"] not in anchor_ids:
                neighbors[records[j]["chunk_id"]] = records[j]
    neighbor_rows = list(neighbors.values())
    if neighbor_rows:
        scores = np.asarray(reranker.predict([(query, r["dense_text"]) for r in neighbor_rows], prompt_name="research",
                                             batch_size=4, show_progress_bar=False, convert_to_numpy=True)).reshape(-1)
        neighbor_rows = [r for _, r in sorted(zip(scores, neighbor_rows), key=lambda x: -x[0])[:TEXT_CFG["neighbor_budget"]]]
    selected = [a["record"] for a in anchors] + neighbor_rows
    seen, evidence = set(), []
    for r in selected:
        key = clean_text(unicodedata.normalize("NFKC", r["source_text"]))
        if key in seen: continue
        seen.add(key)
        evidence.append({
            "source_object_id": r["chunk_id"], "pdf_page_number": r["page_start"] + 1, "modality": "text",
            "content": {"text": r["source_text"], "section_path": r["section_path"]},
            "provenance": {"source_object_ids": r["source_object_ids"], "page_range": [r["page_start"] + 1, r["page_end"] + 1]},
        })
        if len(evidence) >= top_k: break
    return evidence

# 5. Math RAG

Final Math lane: **canonical equations → source graph → equation cards → dense + lexical + source-context retrieval → formula-role verification → exact/source-aware resolution**.

## 5.1 Equation Records and Source Graph

In [22]:
SOURCE_TAG_RE = re.compile(r"\\tag\*?\s*\{\s*([^{}]+?)\s*\}", re.I)
STYLE_RE = re.compile(r"\\(?:mathbf|mathrm|mathit|mathsf|mathtt|boldsymbol|bm)\s*\{([^{}]*)\}")
MATH_EXCLUDED_SECTIONS = {"index", "bibliography", "references", "contents", "table of contents", "attribution"}
MATH_CONTEXT_TYPES = {"paragraph", "list", "definition", "theorem", "proof", "worked_example", "algorithm", "footnote"}

def strip_equation_tag(latex): return SOURCE_TAG_RE.sub("", unicodedata.normalize("NFKC", str(latex or ""))).strip()
def source_equation_label(latex):
    m = SOURCE_TAG_RE.search(str(latex or "")); return f"equation:{m.group(1).strip()}" if m else None

def normalize_formula(latex):
    text = strip_equation_tag(latex)
    for token in ("\\left", "\\right", "\\,", "\\;", "\\!", "\\quad", "\\qquad"): text = text.replace(token, "")
    for _ in range(4):
        newer = STYLE_RE.sub(r"\1", text)
        if newer == text: break
        text = newer
    return re.sub(r"\s+", "", text).strip()

def math_allowed(obj):
    return not any(str(x).strip().lower() in MATH_EXCLUDED_SECTIONS for x in obj.get("section_path") or [])

EQUATION_OBJECTS = sorted([o for o in OBJECTS if o["object_type"] == "equation" and o.get("indexable", True) and clean_text(o.get("latex")) and math_allowed(o)],
                          key=lambda o: (o["page_index"], o["reading_order"]))
MATH_RECORDS = []
for row_id, eq in enumerate(EQUATION_OBJECTS):
    MATH_RECORDS.append({
        "row_id": row_id, "object_id": eq["object_id"], "document_id": DOCUMENT_ID,
        "equation_label": eq.get("label") or source_equation_label(eq.get("latex")), "original_latex": eq["latex"],
        "normalized_formula": normalize_formula(eq["latex"]), "section_path": list(eq.get("section_path") or []),
        "page_index": eq["page_index"], "pdf_page_number": eq["pdf_page_number"], "printed_page_label": eq.get("printed_page_label"),
        "reading_order": eq["reading_order"], "bbox_pdf": eq.get("bbox_pdf"), "previous_id": eq.get("previous_id"), "next_id": eq.get("next_id"),
    })
MATH_BY_OBJECT = {r["object_id"]: r for r in MATH_RECORDS}
MATH_BY_LABEL = {r["equation_label"].lower(): r for r in MATH_RECORDS if r.get("equation_label")}

EQ_NUMBER_RE = re.compile(r"[A-Za-z0-9]+(?:\.[A-Za-z0-9]+)+")
def equation_labels_in_text(text):
    labels = []
    for number in EQ_NUMBER_RE.findall(str(text or "")):
        key = f"equation:{number}".lower()
        if key in MATH_BY_LABEL and key not in labels: labels.append(key)
    return labels

MATH_SOURCE_EDGES, seen_edges = [], set()
def add_math_edge(chunk, equation_object_id, edge_type, weight):
    key = (chunk["chunk_id"], equation_object_id, edge_type)
    if equation_object_id in MATH_BY_OBJECT and key not in seen_edges:
        seen_edges.add(key)
        MATH_SOURCE_EDGES.append({"chunk_id": chunk["chunk_id"], "equation_object_id": equation_object_id, "edge_type": edge_type, "weight": weight})

for chunk in TEXT_RECORDS:
    for oid in chunk.get("references", []): add_math_edge(chunk, oid, "explicit_object_reference", 1.0)
    for label in equation_labels_in_text(chunk.get("source_text", "")): add_math_edge(chunk, MATH_BY_LABEL[label]["object_id"], "textual_equation_reference", 0.98)
    for oid in chunk.get("bridge_object_ids", []): add_math_edge(chunk, oid, "direct_bridge", 0.85)

EQ_TO_SOURCE = defaultdict(list); CHUNK_TO_MATH = defaultdict(list)
for edge in MATH_SOURCE_EDGES:
    EQ_TO_SOURCE[edge["equation_object_id"]].append(edge); CHUNK_TO_MATH[edge["chunk_id"]].append(edge)

write_jsonl(DIRS["math"] / "equations.jsonl", MATH_RECORDS)
write_jsonl(DIRS["math"] / "source_graph.jsonl", MATH_SOURCE_EDGES)
print("Equations:", len(MATH_RECORDS))

Equations: 602


## 5.2 Equation Cards and Formula Roles

In [23]:
def equation_local_context(record, max_objects=4):
    origin, section = OBJECT_BY_ID[record["object_id"]], tuple(record.get("section_path") or [])
    contexts = []
    for pointer in ("previous_id", "next_id"):
        oid, hops = origin.get(pointer), 0
        while oid and hops < 12 and len(contexts) < max_objects:
            obj = OBJECT_BY_ID.get(oid)
            if not obj or tuple(obj.get("section_path") or []) != section or obj["object_type"] == "equation": break
            if obj["object_type"] in MATH_CONTEXT_TYPES and clean_text(obj.get("content")): contexts.append(clean_text(obj["content"]))
            oid, hops = obj.get(pointer), hops + 1
    for edge in EQ_TO_SOURCE.get(record["object_id"], [])[:3]:
        chunk = TEXT_BY_CHUNK.get(edge["chunk_id"])
        if chunk: contexts.append(clean_text(chunk["source_text"]))
    return list(dict.fromkeys(contexts))[:6]

def formula_roles(card):
    latex = str(card.get("original_latex") or "")
    compact = re.sub(r"\s+", "", strip_equation_tag(latex))
    lower = compact.lower()
    context = (" ".join(card.get("source_contexts", [])) + " " + " > ".join(card.get("section_path", []))).lower()
    lhs = compact.split("=", 1)[0] if "=" in compact else compact
    roles = set()

    has_softmax = "softmax" in lower
    has_sigmoid = "sigmoid" in lower or "sig[" in lower or "sig(" in lower
    has_probability = bool(re.search(r"(?:\\Pr|Pr|P\s*r)\s*\(", latex))
    has_partial = r"\partial" in latex
    has_det = bool(re.search(r"\\det\b", latex, re.I) or (has_partial and (r"\left|" in latex or re.search(r"\|\s*\\frac\s*\{\s*\\partial", latex))))
    has_inverse = bool(re.search(r"\^\s*\{\s*-\s*1\s*\}", latex))
    has_log, has_sum = r"\log" in latex, r"\sum" in latex
    has_kl = bool(re.search(r"\bKL\b", latex, re.I))
    has_argmin, has_sqrt = "argmin" in lower, r"\sqrt" in latex
    is_loss = bool(re.search(r"(?:^|[^A-Za-z])L\s*[\[\(\{]", latex) or has_argmin or
                   (has_sum and has_log and ("negative log" in context or "loss" in context)))

    if has_softmax: roles.add("softmax")
    if has_sigmoid: roles.add("sigmoid")
    if has_probability: roles.add("probability")
    if is_loss: roles.add("loss")
    if has_softmax and not has_probability and not is_loss and r"\exp" in latex: roles.add("standalone_softmax")
    if re.search(r"\\(?:mathbf|boldsymbol)\{k\}|(?:^|[^A-Za-z])k[_\{\[]", lhs): roles.add("key_projection")
    if re.search(r"(?:^|[^A-Za-z])a(?:_|\[|\{)", lhs) and (has_softmax or r"\exp" in latex): roles.add("attention_weights")
    if has_sum and ("value vector" in context or "weighted sum" in context): roles.add("weighted_value_sum")
    if has_sqrt and (has_softmax or r"\exp" in latex) and "attention" in context: roles.add("scaled_attention")
    if "matrix form" in context and "attention" in context: roles.add("matrix_attention")
    if has_partial and has_det: roles.add("jacobian_determinant")
    elif has_partial and not has_log and not has_kl: roles.add("jacobian_matrix")

    simple_mapping = "=" in compact and not any((has_partial, has_probability, has_log, has_kl, has_argmin))
    forward_shape = bool(re.search(r"(?:\\mathbf\{)?x(?:\})?=.*(?:\\mathbf\{)?f(?:\})?.*(?:\\mathbf\{)?z(?:\})?", compact, re.I))
    if simple_mapping and forward_shape and not has_inverse: roles.add("forward_mapping")
    if simple_mapping and has_inverse: roles.add("inverse_mapping")
    if "coupling" in context and simple_mapping: roles.add("coupling_inverse" if has_inverse else "coupling_forward")
    if has_probability and has_partial and has_det: roles.add("change_of_variables_density")
    if has_kl and (has_argmin or is_loss): roles.add("kl_objective")
    if has_log and has_det and not has_kl and (has_argmin or is_loss or "maximum likelihood" in context): roles.add("logdet_objective")
    return roles

def build_equation_card(record):
    contexts = equation_local_context(record)
    section = " > ".join(record.get("section_path") or [])
    semantic = "\n".join(x for x in (f"Section: {section}" if section else "", *contexts[:4], strip_equation_tag(record["original_latex"])) if x)
    lexical = "\n".join(x for x in ((record.get("equation_label") or "").replace(":", " "), section, *contexts, strip_equation_tag(record["original_latex"])) if x)
    card = dict(record, source_contexts=contexts, semantic_text=semantic, lexical_text=lexical)
    card["formula_roles"] = sorted(formula_roles(card))
    return card

EQUATION_CARDS = [build_equation_card(r) for r in MATH_RECORDS]
MATH_CARD_BY_ROW = {c["row_id"]: c for c in EQUATION_CARDS}
MATH_CARD_BY_OBJECT = {c["object_id"]: c for c in EQUATION_CARDS}
MATH_CARD_BY_LABEL = {c["equation_label"].lower(): c for c in EQUATION_CARDS if c.get("equation_label")}
write_jsonl(DIRS["math"] / "equation_cards.jsonl", EQUATION_CARDS)

## 5.3 Math Indexes and Resolver

In [24]:
math_vectors = embedder.encode([c["semantic_text"] for c in EQUATION_CARDS], batch_size=16, normalize_embeddings=True,
                               convert_to_numpy=True, show_progress_bar=True).astype("float32")
math_vectors = np.ascontiguousarray(math_vectors); faiss.normalize_L2(math_vectors)
MATH_DENSE_INDEX = faiss.IndexFlatIP(math_vectors.shape[1]); MATH_DENSE_INDEX.add(math_vectors)
faiss.write_index(MATH_DENSE_INDEX, str(DIRS["math"] / "dense.faiss"))

math_bm25_tokenizer = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None)
math_tokens = math_bm25_tokenizer.tokenize([c["lexical_text"] for c in EQUATION_CARDS], return_as="tuple")
MATH_BM25 = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); MATH_BM25.index(math_tokens)
MATH_BM25_DIR = DIRS["math"] / "bm25"; MATH_BM25_DIR.mkdir(parents=True, exist_ok=True)
MATH_BM25.save(str(MATH_BM25_DIR)); math_bm25_tokenizer.save_vocab(save_dir=str(MATH_BM25_DIR)); math_bm25_tokenizer.save_stopwords(save_dir=str(MATH_BM25_DIR))

def math_dense_search(query, k=None):
    k = min(k or MATH_CFG["dense_k"], len(EQUATION_CARDS)); scores, rows = MATH_DENSE_INDEX.search(encode_query(query), k)
    return [{"id": MATH_CARD_BY_ROW[int(row)]["object_id"], "rank": i, "score": float(score), "card": MATH_CARD_BY_ROW[int(row)]}
            for i, (row, score) in enumerate(zip(rows[0], scores[0]), 1) if row >= 0]

def math_bm25_search(query, k=None):
    k = min(k or MATH_CFG["bm25_k"], len(EQUATION_CARDS)); tokens = math_bm25_tokenizer.tokenize([query], update_vocab=False, return_as="tuple")
    rows, scores = MATH_BM25.retrieve(tokens, k=k, return_as="tuple", show_progress=False)
    hits = [{"id": MATH_CARD_BY_ROW[int(row)]["object_id"], "score": float(score), "card": MATH_CARD_BY_ROW[int(row)]}
            for row, score in zip(rows[0], scores[0]) if row >= 0 and float(score) > 0]
    for i, hit in enumerate(hits, 1): hit["rank"] = i
    return hits

def source_equation_search(query):
    text_ranked = rerank_text(query, text_rrf(query))[:MATH_CFG["source_chunk_k"]]
    scores = defaultdict(float)
    for chunk_rank, hit in enumerate(text_ranked, 1):
        for edge in CHUNK_TO_MATH.get(hit["record"]["chunk_id"], []):
            scores[edge["equation_object_id"]] += edge["weight"] / (MATH_CFG["rrf_k"] + chunk_rank)
    ranked = sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:MATH_CFG["source_equation_k"]]
    return [{"id": oid, "rank": i, "score": float(score), "card": MATH_CARD_BY_OBJECT[oid]} for i, (oid, score) in enumerate(ranked, 1)]

def math_candidates(query):
    lanes = {"dense": math_dense_search(query), "bm25": math_bm25_search(query), "source": source_equation_search(query)}
    fused = weighted_rrf(lanes, {"dense": 1.0, "bm25": 1.0, "source": 1.0}, MATH_CFG["rrf_k"], MATH_CFG["rerank_pool"])
    cards = {x["id"]: x["card"] for rows in lanes.values() for x in rows}
    candidates = [dict(item, card=cards[item["id"]]) for item in fused]
    if not candidates: return []
    pairs = [(query, "\n".join(["Section: " + " > ".join(c["card"]["section_path"]),
                                  "Context: " + " ".join(c["card"].get("source_contexts", [])[:2]),
                                  "Formula: " + strip_equation_tag(c["card"]["original_latex"]),
                                  "Roles: " + ", ".join(c["card"].get("formula_roles", []))])) for c in candidates]
    scores = np.asarray(reranker.predict(pairs, prompt_name="math_equation", batch_size=4, show_progress_bar=False, convert_to_numpy=True)).reshape(-1)
    ranked = [dict(c, rerank_score=float(score)) for c, score in zip(candidates, scores)]
    ranked.sort(key=lambda x: (-x["rerank_score"], x["rank"]))
    return ranked

EXPLICIT_EQ_RE = re.compile(r"\b(?:eq(?:uation)?s?\.?|formula)\s*\(?\s*([A-Za-z]\.\d+|\d+(?:\.\d+)+)\s*\)?", re.I)

def requested_math_roles(query):
    q = clean_text(query).lower(); required, forbidden = set(), set()
    if "softmax" in q and any(x in q for x in ("define", "definition", "standalone", "expression")): required |= {"softmax", "standalone_softmax"}; forbidden |= {"loss"}
    if "softmax" in q and any(x in q for x in ("probability", "probabilities", "class logits", "normalized")): required.add("softmax")
    if "jacobian" in q and "determinant" in q: required.add("jacobian_determinant"); forbidden.add("jacobian_matrix")
    elif "jacobian" in q or "derivative" in q: required.add("jacobian_matrix"); forbidden.add("jacobian_determinant")
    if "inverse" in q and "mapping" in q: required.add("inverse_mapping"); forbidden.add("forward_mapping")
    elif "forward" in q and "mapping" in q: required.add("forward_mapping"); forbidden.add("inverse_mapping")
    if "attention" in q and "weight" in q: required.add("attention_weights")
    if "weighted sum" in q and "value" in q: required.add("weighted_value_sum")
    return required, forbidden

def equation_payload(card):
    return {"source_object_id": card["object_id"], "pdf_page_number": card["pdf_page_number"], "modality": "math",
            "content": {"latex": card["original_latex"], "equation_label": card.get("equation_label"), "section_path": card["section_path"]},
            "provenance": {"bbox_pdf": card.get("bbox_pdf"), "source_contexts": card.get("source_contexts", [])[:2]}}

def retrieve_math(query, top_k=3):
    labels = [f"equation:{x}".lower() for x in EXPLICIT_EQ_RE.findall(query)]
    if labels:
        cards = [MATH_CARD_BY_LABEL[label] for label in labels if label in MATH_CARD_BY_LABEL]
        return [equation_payload(c) for c in cards]
    required, forbidden = requested_math_roles(query)
    ranked = math_candidates(query)
    selected = []
    for item in ranked:
        roles = set(item["card"].get("formula_roles", []))
        if forbidden & roles: continue
        if required and not required.issubset(roles): continue
        selected.append(equation_payload(item["card"]))
        if len(selected) >= top_k: break
    if not selected and not required and ranked:
        selected = [equation_payload(ranked[0]["card"])]
    return selected

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/602 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/602 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/602 [00:00<?, ?it/s]

# 6. Visual RAG

Final Visual lane: **source-bound crops + caption/context representation + Qwen semantic retrieval + BM25 source retrieval + multi-view SigLIP2 + optional OCR lexical auxiliary → query-aware weighted RRF → group-aware evidence**.

OCR is auxiliary, not authoritative. The original PDF crop and canonical caption remain the evidence source.

## 6.1 Visual Registry and Source Crops

In [25]:
VISUAL_TYPES = {"figure", "chart"}
VISUAL_OBJECTS = sorted([o for o in OBJECTS if o["object_type"] in VISUAL_TYPES and o.get("bbox_pdf")], key=lambda o: (o["page_index"], o["reading_order"]))
VISUAL_CROPS_DIR = DIRS["visual"] / "crops"; VISUAL_CROPS_DIR.mkdir(parents=True, exist_ok=True)

def nearby_text(obj, radius=2):
    same_page = [o for o in ordered if o["page_index"] == obj["page_index"]]
    pos = next((i for i, o in enumerate(same_page) if o["object_id"] == obj["object_id"]), None)
    if pos is None: return ""
    nearby = []
    for other in same_page[max(0, pos - radius): pos + radius + 1]:
        if other["object_id"] != obj["object_id"] and other["object_type"] in TEXT_TYPES and clean_text(other.get("content")):
            nearby.append(clean_text(other["content"]))
    return " ".join(nearby)

def render_crop(obj, scale=2.0):
    page = pdf.load_page(obj["page_index"])
    pix = page.get_pixmap(matrix=pymupdf.Matrix(scale, scale), clip=pymupdf.Rect(obj["bbox_pdf"]), alpha=False)
    path = VISUAL_CROPS_DIR / f"{stable_hash(obj['object_id'], n=16)}.png"
    pix.save(str(path))
    return path

def crop_visible_text(obj):
    page = pdf.load_page(obj["page_index"])
    words = page.get_text("words", clip=pymupdf.Rect(obj["bbox_pdf"]))
    return clean_text(" ".join(w[4] for w in sorted(words, key=lambda w: (w[1], w[0]))))

VISUAL_RECORDS = []
for row_id, obj in enumerate(VISUAL_OBJECTS):
    crop_path = render_crop(obj)
    label = obj.get("label")
    reference_key = label.split(":", 1)[1].lower() if label and ":" in label else None
    source_text = "\n".join(x for x in (obj.get("caption") or "", nearby_text(obj)) if x)
    visible_text = crop_visible_text(obj)
    VISUAL_RECORDS.append({
        "row_id": row_id, "visual_id": obj["object_id"], "document_id": DOCUMENT_ID, "visual_type": obj["object_type"],
        "pdf_page_number": obj["pdf_page_number"], "section_path": obj.get("section_path") or [], "bbox_pdf": obj["bbox_pdf"],
        "label": label, "reference_key": reference_key, "caption": obj.get("caption"), "source_text": source_text,
        "visible_text": visible_text, "ocr_text": "", "crop_path": str(crop_path), "crop_sha256": sha256_file(crop_path),
        "visual_group_id": label.lower() if label else None,
    })
VISUAL_BY_ID = {r["visual_id"]: r for r in VISUAL_RECORDS}
write_jsonl(DIRS["visual"] / "records.jsonl", VISUAL_RECORDS)
print("Visual records:", len(VISUAL_RECORDS))

Visual records: 313


## 6.2 Optional OCR Auxiliary

Enable only when visual text that is not available in the PDF text layer materially helps retrieval. OCR output is never treated as source authority.

In [26]:
if ENABLE_AUX_OCR:
    import easyocr
    ocr_reader = easyocr.Reader(["en"], gpu=(DEVICE == "cuda"))
    for row in VISUAL_RECORDS:
        if len(row["visible_text"]) < 30:
            row["ocr_text"] = clean_text(" ".join(ocr_reader.readtext(row["crop_path"], detail=0)))
    write_jsonl(DIRS["visual"] / "records.jsonl", VISUAL_RECORDS)

## 6.3 Visual Text Indexes

In [27]:
VISUAL_TEXT = ["\n".join(x for x in (r.get("label") or "", " > ".join(r["section_path"]), r["source_text"], r["visible_text"]) if x) for r in VISUAL_RECORDS]
visual_vectors = encode_docs(VISUAL_TEXT, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype("float32")
visual_vectors = np.ascontiguousarray(visual_vectors); faiss.normalize_L2(visual_vectors)
VISUAL_QWEN_INDEX = faiss.IndexFlatIP(visual_vectors.shape[1]); VISUAL_QWEN_INDEX.add(visual_vectors)
faiss.write_index(VISUAL_QWEN_INDEX, str(DIRS["visual"] / "qwen.faiss"))

visual_bm25_tokenizer = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None)
visual_tokens = visual_bm25_tokenizer.tokenize(VISUAL_TEXT, return_as="tuple")
VISUAL_BM25 = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); VISUAL_BM25.index(visual_tokens)
VISUAL_BM25_DIR = DIRS["visual"] / "bm25"; VISUAL_BM25_DIR.mkdir(parents=True, exist_ok=True)
VISUAL_BM25.save(str(VISUAL_BM25_DIR)); visual_bm25_tokenizer.save_vocab(save_dir=str(VISUAL_BM25_DIR)); visual_bm25_tokenizer.save_stopwords(save_dir=str(VISUAL_BM25_DIR))

ocr_bm25_tokenizer, VISUAL_OCR_BM25 = None, None
if any(clean_text(r.get("ocr_text")) for r in VISUAL_RECORDS):
    ocr_bm25_tokenizer = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None)
    ocr_tokens = ocr_bm25_tokenizer.tokenize([r.get("ocr_text") or "" for r in VISUAL_RECORDS], return_as="tuple")
    VISUAL_OCR_BM25 = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); VISUAL_OCR_BM25.index(ocr_tokens)
    ocr_dir = DIRS["visual"] / "ocr_bm25"; ocr_dir.mkdir(parents=True, exist_ok=True)
    VISUAL_OCR_BM25.save(str(ocr_dir)); ocr_bm25_tokenizer.save_vocab(save_dir=str(ocr_dir)); ocr_bm25_tokenizer.save_stopwords(save_dir=str(ocr_dir))

REFERENCE_INDEX = defaultdict(list)
for r in VISUAL_RECORDS:
    if r.get("reference_key"): REFERENCE_INDEX[r["reference_key"]].append(r["visual_id"])

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/313 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/313 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/313 [00:00<?, ?it/s]

## 6.4 Multi-View SigLIP2 Index

In [28]:
from transformers import AutoModel, AutoProcessor

SIGLIP_DEVICE = DEVICE
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
siglip_model = AutoModel.from_pretrained(SIGLIP_MODEL).to(SIGLIP_DEVICE).eval()

def visual_views(image):
    w, h = image.size
    views = [("full", [0, 0, w, h], image)]
    if min(w, h) >= 256:
        cw, ch = max(w // 2, 128), max(h // 2, 128)
        boxes = [("top_left", [0, 0, cw, ch]), ("top_right", [w-cw, 0, w, ch]),
                 ("bottom_left", [0, h-ch, cw, h]), ("bottom_right", [w-cw, h-ch, w, h])]
        views.extend((name, box, image.crop(tuple(box))) for name, box in boxes)
    return views

view_meta, view_vectors = [], []
batch_images, batch_meta = [], []
def flush_siglip_batch():
    global batch_images, batch_meta
    if not batch_images: return
    inputs = siglip_processor(images=batch_images, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(SIGLIP_DEVICE)
    with torch.inference_mode():
        feats = siglip_model.get_image_features(pixel_values=pixel_values)
        if hasattr(feats, "pooler_output"): feats = feats.pooler_output
        feats = torch.nn.functional.normalize(feats.float(), dim=-1)
    view_vectors.append(feats.cpu().numpy().astype("float32")); view_meta.extend(batch_meta)
    batch_images, batch_meta = [], []

for row in VISUAL_RECORDS:
    with Image.open(row["crop_path"]) as im:
        rgb = im.convert("RGB")
        for name, box, view in visual_views(rgb):
            batch_images.append(view); batch_meta.append({"visual_id": row["visual_id"], "view_type": name, "bbox_px": box})
            if len(batch_images) >= (24 if DEVICE == "cuda" else 4): flush_siglip_batch()
flush_siglip_batch()

SIGLIP_VIEW_VECTORS = np.concatenate(view_vectors, axis=0) if view_vectors else np.empty((0, 0), dtype="float32")
VIEW_INDICES = defaultdict(list)
for i, meta in enumerate(view_meta): VIEW_INDICES[meta["visual_id"]].append(i)
np.save(DIRS["visual"] / "siglip_multiview.npy", SIGLIP_VIEW_VECTORS)
write_jsonl(DIRS["visual"] / "siglip_multiview_meta.jsonl", view_meta)

def encode_siglip_query(query):
    inputs = siglip_processor(text=[query], padding="max_length", truncation=True, return_tensors="pt")
    text_inputs = {k: v.to(SIGLIP_DEVICE) for k, v in inputs.items() if k in {"input_ids", "attention_mask"}}
    with torch.inference_mode():
        feats = siglip_model.get_text_features(**text_inputs)
        if hasattr(feats, "pooler_output"): feats = feats.pooler_output
        feats = torch.nn.functional.normalize(feats.float(), dim=-1)
    return feats.cpu().numpy()[0].astype("float32")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

## 6.5 Routed Visual Retrieval API

In [29]:
EXACT_VISUAL_RE = re.compile(r"\b(?:figure|fig\.?|chart)\s*[:#]?\s*(\d+(?:\.\d+)*[a-z]?)\b", re.I)
VISUAL_CUE_RE = re.compile(r"\b(diagram|architecture|plot|graph|curve|axis|legend|arrow|panel|image|picture|visual|dashed|dotted|blue|red|green|left|right|top|bottom)\b", re.I)

def classify_visual_query(query):
    m = EXACT_VISUAL_RE.search(query)
    if m: return {"intent": "exact_reference", "reference_key": m.group(1).lower()}
    return {"intent": "visual_semantic" if VISUAL_CUE_RE.search(query) else "semantic", "reference_key": None}

def visual_qwen_search(query, k=30):
    scores, rows = VISUAL_QWEN_INDEX.search(encode_query(query), min(k, len(VISUAL_RECORDS)))
    return [{"id": VISUAL_RECORDS[int(row)]["visual_id"], "rank": i, "score": float(score)} for i, (row, score) in enumerate(zip(rows[0], scores[0]), 1) if row >= 0]

def visual_bm25_search(query, k=30):
    tokens = visual_bm25_tokenizer.tokenize([query], update_vocab=False, return_as="tuple")
    rows, scores = VISUAL_BM25.retrieve(tokens, k=min(k, len(VISUAL_RECORDS)), return_as="tuple", show_progress=False)
    hits = [{"id": VISUAL_RECORDS[int(row)]["visual_id"], "score": float(score)} for row, score in zip(rows[0], scores[0]) if row >= 0 and float(score) > 0]
    for i, hit in enumerate(hits, 1): hit["rank"] = i
    return hits

def visual_ocr_search(query, k=30):
    if VISUAL_OCR_BM25 is None: return []
    tokens = ocr_bm25_tokenizer.tokenize([query], update_vocab=False, return_as="tuple")
    rows, scores = VISUAL_OCR_BM25.retrieve(tokens, k=min(k, len(VISUAL_RECORDS)), return_as="tuple", show_progress=False)
    hits = [{"id": VISUAL_RECORDS[int(row)]["visual_id"], "score": float(score)} for row, score in zip(rows[0], scores[0]) if row >= 0 and float(score) > 0]
    for i, hit in enumerate(hits, 1): hit["rank"] = i
    return hits

def siglip_search(query, k=30):
    if SIGLIP_VIEW_VECTORS.size == 0: return []
    q, rows = encode_siglip_query(query), []
    scores = SIGLIP_VIEW_VECTORS @ q
    for visual_id, idxs in VIEW_INDICES.items():
        best = max(idxs, key=lambda i: scores[i]); rows.append((visual_id, float(scores[best])))
    rows.sort(key=lambda x: (-x[1], x[0]))
    return [{"id": vid, "rank": i, "score": score} for i, (vid, score) in enumerate(rows[:k], 1)]

def visual_rank(query):
    route = classify_visual_query(query)
    if route["intent"] == "exact_reference":
        direct = REFERENCE_INDEX.get(route["reference_key"], [])
        return [{"id": vid, "rank": i, "score": 1.0, "lanes": {"exact_reference": {"rank": i}}} for i, vid in enumerate(direct, 1)]
    lanes = {"qwen_source": visual_qwen_search(query), "bm25_source": visual_bm25_search(query), "ocr_lexical": visual_ocr_search(query)}
    weights = VISUAL_CFG["semantic_weights"]
    if route["intent"] == "visual_semantic":
        lanes["siglip_multiview"] = siglip_search(query); weights = VISUAL_CFG["visual_weights"]
    return weighted_rrf(lanes, weights, VISUAL_CFG["rrf_k"], VISUAL_CFG["candidate_k"])


VISUAL_STOPWORDS = {"find", "show", "locate", "figure", "fig", "chart", "diagram", "plot", "graph", "image", "visual", "architecture", "the", "a", "an", "of", "in", "on", "with", "and", "to"}

def visual_subject_terms(query):
    return [t for t in technical_split(query) if len(t) > 2 and t not in VISUAL_STOPWORDS]

def rerank_visual_candidates(query, candidates):
    pool = candidates[:VISUAL_CFG["rerank_pool"]]
    if not pool or not visual_subject_terms(query): return candidates
    pairs = []
    for item in pool:
        row = VISUAL_BY_ID[item["id"]]
        text = "\n".join(x for x in (row.get("label") or "", row.get("caption") or "", row.get("source_text") or "", row.get("visible_text") or "") if x)
        pairs.append((query, text))
    scores = np.asarray(reranker.predict(pairs, prompt_name="research", batch_size=4, show_progress_bar=False, convert_to_numpy=True)).reshape(-1)
    rescored = [dict(item, relevance_score=float(score)) for item, score in zip(pool, scores)]
    rescored.sort(key=lambda x: (-x["relevance_score"], x["rank"]))
    for rank, item in enumerate(rescored, 1): item["rank"] = rank
    ids = {x["id"] for x in rescored}
    return rescored + [x for x in candidates if x["id"] not in ids]

def retrieve_visual(query, top_k=None):
    top_k = top_k or VISUAL_CFG["evidence_k"]
    selected, used_groups = [], set()
    ranked = visual_rank(query)
    if classify_visual_query(query)["intent"] != "exact_reference": ranked = rerank_visual_candidates(query, ranked)
    for item in ranked:
        row = VISUAL_BY_ID[item["id"]]; group = row.get("visual_group_id") or row["visual_id"]
        if group in used_groups: continue
        used_groups.add(group)
        selected.append({
            "source_object_id": row["visual_id"], "pdf_page_number": row["pdf_page_number"], "modality": "visual",
            "content": {"label": row.get("label"), "caption": row.get("caption"), "visual_type": row["visual_type"], "crop_path": row["crop_path"]},
            "provenance": {"bbox_pdf": row["bbox_pdf"], "crop_sha256": row["crop_sha256"], "retrieval_lanes": item.get("lanes", {})},
        })
        if len(selected) >= top_k: break
    return selected

# 7. Table RAG

Final Table lane: **canonical table objects → structured cells/rows → typed values + header mapping → table- and row-level Qwen/BM25 retrieval → source-bound cell evidence**.

## 7.1 Structured Table Records

In [30]:
def numeric_value(text):
    raw = clean_text(text).replace(",", "")
    if not raw: return None
    m = re.fullmatch(r"([+-]?\d+(?:\.\d+)?)\s*(%|[A-Za-zµμ/$]+)?", raw)
    if not m: return None
    return {"value": float(m.group(1)), "unit": m.group(2) or None, "raw": clean_text(text)}

def expand_table_cells(cells):
    expanded = []
    for cell in cells:
        for r in range(cell["row"], cell["row"] + cell.get("rowspan", 1)):
            for c in range(cell["col"], cell["col"] + cell.get("colspan", 1)):
                expanded.append({"row": r, "col": c, "text": cell["text"], "is_header": cell.get("is_header", False),
                                 "origin_row": cell["row"], "origin_col": cell["col"]})
    return expanded

def pymupdf_table_cells(obj):
    try:
        page = pdf.load_page(obj["page_index"])
        finder = page.find_tables(clip=pymupdf.Rect(obj["bbox_pdf"]))
        if not finder.tables: return []
        data = finder.tables[0].extract()
        return [{"row": r, "col": c, "text": clean_text(value), "is_header": r == 0, "origin_row": r, "origin_col": c}
                for r, row in enumerate(data) for c, value in enumerate(row) if clean_text(value)]
    except Exception:
        return []

def table_quality(cells):
    if not cells: return {"accepted": False, "reason": "no_cells", "rows": 0, "cols": 0, "density": 0.0}
    rows = max(c["row"] for c in cells) + 1; cols = max(c["col"] for c in cells) + 1
    nonempty = sum(bool(clean_text(c.get("text"))) for c in cells)
    density = nonempty / max(rows * cols, 1)
    accepted = rows >= TABLE_CFG["min_rows"] and cols >= TABLE_CFG["min_cols"] and nonempty >= TABLE_CFG["min_nonempty"] and density >= TABLE_CFG["min_density"]
    reason = "accepted" if accepted else "insufficient_structure"
    return {"accepted": accepted, "reason": reason, "rows": rows, "cols": cols, "nonempty": nonempty, "density": round(density, 4)}

TABLE_OBJECTS = sorted([o for o in OBJECTS if o["object_type"] == "table" and o.get("bbox_pdf")], key=lambda o: (o["page_index"], o["reading_order"]))
TABLE_RECORDS, TABLE_CELL_RECORDS, TABLE_ROW_RECORDS, TABLE_CANDIDATES = [], [], [], []

for table_idx, obj in enumerate(TABLE_OBJECTS):
    cells = expand_table_cells(obj.get("table_cells") or []) or pymupdf_table_cells(obj)
    quality = table_quality(cells)
    TABLE_CANDIDATES.append({"table_id": obj["object_id"], "pdf_page_number": obj["pdf_page_number"], "label": obj.get("label"), "caption": obj.get("caption"), **quality})
    if not quality["accepted"]: continue
    max_row, max_col = max(c["row"] for c in cells), max(c["col"] for c in cells)
    cell_map = {(c["row"], c["col"]): c for c in cells}
    headers = {c: cell_map.get((0, c), {}).get("text", "") for c in range(max_col + 1)}
    table_id = obj["object_id"]
    table_cells = []
    for c in cells:
        cid = f"{table_id}:r{c['row']:04d}:c{c['col']:04d}"
        record = {
            "cell_id": cid, "table_id": table_id, "row": c["row"], "col": c["col"], "text": c["text"],
            "header": headers.get(c["col"]) or None, "is_header": c["is_header"], "typed_value": numeric_value(c["text"]),
            "pdf_page_number": obj["pdf_page_number"], "bbox_pdf": obj["bbox_pdf"],
        }
        TABLE_CELL_RECORDS.append(record); table_cells.append(record)
    row_ids = []
    for r in range(max_row + 1):
        values = [cell_map.get((r, c), {}).get("text", "") for c in range(max_col + 1)]
        row_text = " | ".join(x for x in values if x)
        if not row_text: continue
        rid = f"{table_id}:row:{r:04d}"; row_ids.append(rid)
        TABLE_ROW_RECORDS.append({"row_id": rid, "table_id": table_id, "row": r, "text": row_text,
                                  "values": values, "pdf_page_number": obj["pdf_page_number"], "section_path": obj.get("section_path") or []})
    caption = obj.get("caption") or obj.get("content") or ""
    table_text = "\n".join([caption, " > ".join(obj.get("section_path") or []), *[r["text"] for r in TABLE_ROW_RECORDS if r["table_id"] == table_id]])
    TABLE_RECORDS.append({"table_id": table_id, "row_id": table_idx, "label": obj.get("label"), "caption": caption,
                          "pdf_page_number": obj["pdf_page_number"], "section_path": obj.get("section_path") or [], "bbox_pdf": obj["bbox_pdf"],
                          "table_text": table_text, "cell_ids": [c["cell_id"] for c in table_cells], "row_ids": row_ids, "reconstruction": "structured"})

TABLE_BY_ID = {t["table_id"]: t for t in TABLE_RECORDS}
TABLE_CELL_BY_ID = {c["cell_id"]: c for c in TABLE_CELL_RECORDS}
TABLE_ROW_BY_ID = {r["row_id"]: r for r in TABLE_ROW_RECORDS}
write_jsonl(DIRS["table"] / "candidates.jsonl", TABLE_CANDIDATES)
write_jsonl(DIRS["table"] / "tables.jsonl", TABLE_RECORDS)
write_jsonl(DIRS["table"] / "rows.jsonl", TABLE_ROW_RECORDS)
write_jsonl(DIRS["table"] / "cells.jsonl", TABLE_CELL_RECORDS)
print("Table candidates:", len(TABLE_CANDIDATES), "| Accepted:", len(TABLE_RECORDS), "| Rows:", len(TABLE_ROW_RECORDS), "| Cells:", len(TABLE_CELL_RECORDS))

Table candidates: 4 | Accepted: 4 | Rows: 23 | Cells: 115


## 7.2 Hybrid Table and Row Indexes

In [31]:
def build_hybrid_index(texts, out_dir, prefix):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    if not texts: return None, None, None
    vectors = encode_docs(texts, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype("float32")
    vectors = np.ascontiguousarray(vectors); faiss.normalize_L2(vectors)
    dense = faiss.IndexFlatIP(vectors.shape[1]); dense.add(vectors); faiss.write_index(dense, str(out_dir / f"{prefix}.faiss"))
    tok = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None); tokens = tok.tokenize(texts, return_as="tuple")
    bm = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); bm.index(tokens)
    bm_dir = out_dir / f"{prefix}_bm25"; bm_dir.mkdir(parents=True, exist_ok=True)
    bm.save(str(bm_dir)); tok.save_vocab(save_dir=str(bm_dir)); tok.save_stopwords(save_dir=str(bm_dir))
    return dense, bm, tok

TABLE_DENSE, TABLE_BM25, table_bm25_tokenizer = build_hybrid_index([t["table_text"] for t in TABLE_RECORDS], DIRS["table"], "tables")
ROW_DENSE, ROW_BM25, row_bm25_tokenizer = build_hybrid_index([r["text"] for r in TABLE_ROW_RECORDS], DIRS["table"], "rows")

def hybrid_search_records(query, records, dense_index, bm25_index, tokenizer, id_field, k=20):
    if not records or dense_index is None or bm25_index is None or tokenizer is None: return []
    k = min(k, len(records)); scores, rows = dense_index.search(encode_query(query), k)
    dense = [{"id": records[int(row)][id_field], "rank": i, "score": float(score)} for i, (row, score) in enumerate(zip(rows[0], scores[0]), 1) if row >= 0]
    qtok = tokenizer.tokenize([query], update_vocab=False, return_as="tuple"); brow, bscores = bm25_index.retrieve(qtok, k=k, return_as="tuple", show_progress=False)
    lexical = [{"id": records[int(row)][id_field], "score": float(score)} for row, score in zip(brow[0], bscores[0]) if row >= 0 and float(score) > 0]
    for i, hit in enumerate(lexical, 1): hit["rank"] = i
    return weighted_rrf({"dense": dense, "bm25": lexical}, {"dense": 1.0, "bm25": 1.0}, TABLE_CFG["rrf_k"], k)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/4 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/4 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/23 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/23 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/23 [00:00<?, ?it/s]

## 7.3 Table Retrieval API

In [32]:
TABLE_REFERENCE_RE = re.compile(r"\btable\s*(\d+(?:\.\d+)*)", re.I)
PAGE_REFERENCE_RE = re.compile(r"\b(?:pdf\s*)?page\s*(\d+)\b", re.I)

def retrieve_table(query, top_k=None, page_constraint=None):
    top_k = top_k or TABLE_CFG["evidence_k"]
    explicit = TABLE_REFERENCE_RE.search(query)
    tables = []
    if explicit:
        target = f"table:{explicit.group(1)}".lower()
        tables = [t for t in TABLE_RECORDS if (t.get("label") or "").lower() == target]
    elif page_constraint:
        tables = [t for t in TABLE_RECORDS if t["pdf_page_number"] == page_constraint]
    else:
        ranked = hybrid_search_records(query, TABLE_RECORDS, TABLE_DENSE, TABLE_BM25, table_bm25_tokenizer, "table_id", TABLE_CFG["table_k"])
        tables = [TABLE_BY_ID[x["id"]] for x in ranked]

    row_ranked = hybrid_search_records(query, TABLE_ROW_RECORDS, ROW_DENSE, ROW_BM25, row_bm25_tokenizer, "row_id", TABLE_CFG["row_k"])
    row_by_table = defaultdict(list)
    for item in row_ranked: row_by_table[TABLE_ROW_BY_ID[item["id"]]["table_id"]].append(TABLE_ROW_BY_ID[item["id"]])

    evidence = []
    for table in tables:
        selected_rows = row_by_table.get(table["table_id"], [])[:3]
        if not selected_rows:
            selected_rows = [TABLE_ROW_BY_ID[rid] for rid in table["row_ids"][:3]]
        selected_cells = [c for c in TABLE_CELL_RECORDS if c["table_id"] == table["table_id"] and c["row"] in {r["row"] for r in selected_rows}]
        evidence.append({
            "source_object_id": table["table_id"], "pdf_page_number": table["pdf_page_number"], "modality": "table",
            "content": {"label": table.get("label"), "caption": table.get("caption"), "rows": selected_rows,
                        "selected_cells": [{"cell_id": c["cell_id"], "row": c["row"], "col": c["col"], "header": c.get("header"), "raw_value": c["text"], "typed_value": c.get("typed_value")} for c in selected_cells]},
            "provenance": {"bbox_pdf": table["bbox_pdf"], "reconstruction": table["reconstruction"]},
        })
        if len(evidence) >= top_k: break
    return evidence

# 8. Query Planner

The planner performs deterministic normalization, explicit reference parsing, safety routing, modality detection, and lightweight compound decomposition. It never fabricates source references.

In [33]:
NUMERIC_OPERATION_RE = re.compile(r"\b(?:average|mean|sum|total|percentage|percent|difference|ratio|calculate|compute)\b", re.I)
POLICY_BYPASS_RE = re.compile(r"\b(?:ignore previous|fabricate citations?|invent citations?|fake citations?)\b", re.I)
FIGURE_WORD_RE = re.compile(r"\b(?:figure|fig\.?|diagram|plot|graph|chart|architecture|image|visual)\b", re.I)
TABLE_WORD_RE = re.compile(r"\b(?:table|row|column|cell|coverage)\b", re.I)
MATH_WORD_RE = re.compile(r"\b(?:eq(?:uation)?s?\.?|formula|mathematical expression|derive|derivative|jacobian|softmax|sigmoid|logits?)\b", re.I)
TEXT_WORD_RE = re.compile(r"\b(?:why|explain|describe|discuss|what does|how does|meaning|represent)\b", re.I)

def parse_references(query):
    pages = [int(x) for x in PAGE_REFERENCE_RE.findall(query)]
    figures = [x.lower() for x in EXACT_VISUAL_RE.findall(query)]
    tables = [x.lower() for x in TABLE_REFERENCE_RE.findall(query)]
    equations = [x.lower() for x in EXPLICIT_EQ_RE.findall(query)]
    return {"pdf_pages": pages, "figures": figures, "tables": tables, "equations": equations}

def detect_modalities(query):
    mods = []
    if TABLE_WORD_RE.search(query): mods.append("table")
    if FIGURE_WORD_RE.search(query): mods.append("visual")
    if MATH_WORD_RE.search(query): mods.append("math")
    if TEXT_WORD_RE.search(query) or not mods: mods.append("text")
    return list(dict.fromkeys(mods))

def decompose_query(query, modalities):
    if len(modalities) <= 1: return [{"subquery_id": "sq_001", "query": query, "modalities": modalities}]
    return [{"subquery_id": f"sq_{i:03d}", "query": query, "modalities": [modality]} for i, modality in enumerate(modalities, 1)]

def unresolved_references(refs):
    missing = []
    known_figures = set(REFERENCE_INDEX)
    known_tables = {(t.get("label") or "").split(":", 1)[-1].lower() for t in TABLE_RECORDS if t.get("label")}
    known_equations = {(c.get("equation_label") or "").split(":", 1)[-1].lower() for c in EQUATION_CARDS if c.get("equation_label")}
    missing += [f"figure:{x}" for x in refs["figures"] if x not in known_figures]
    missing += [f"table:{x}" for x in refs["tables"] if x not in known_tables]
    missing += [f"equation:{x}" for x in refs["equations"] if x not in known_equations]
    return missing

def plan_query(query):
    query = clean_text(query)
    if not query: return {"status": "invalid_query", "query": query, "modalities": [], "subqueries": [], "references": {}}
    if POLICY_BYPASS_RE.search(query): return {"status": "blocked_policy_bypass", "query": query, "modalities": [], "subqueries": [], "references": {}}
    refs = parse_references(query)
    if any(p < 1 or p > len(PAGES) for p in refs["pdf_pages"]):
        return {"status": "blocked_invalid_reference", "query": query, "modalities": detect_modalities(query), "subqueries": [], "references": refs}
    if len(refs["pdf_pages"]) > 1 or unresolved_references(refs):
        return {"status": "requires_reference_resolution", "query": query, "modalities": detect_modalities(query), "subqueries": [], "references": refs}
    if NUMERIC_OPERATION_RE.search(query) and TABLE_WORD_RE.search(query):
        return {"status": "numeric_engine_required", "query": query, "modalities": ["table"], "subqueries": [], "references": refs}
    modalities = detect_modalities(query)
    return {"status": "plan_ready", "query": query, "modalities": modalities, "subqueries": decompose_query(query, modalities), "references": refs}

# 9. Unified Retrieval

All four lanes emit the same evidence contract. Raw cross-modal similarity scores are never directly compared; each modality ranks internally, then the evidence is fused by subquery and modality order.

In [34]:
MODALITY_ORDER = {"text": 0, "math": 1, "visual": 2, "table": 3}

def evidence_id(item):
    return f"{DOCUMENT_ID}:{item['modality']}:{stable_hash(item['source_object_id'], item['pdf_page_number'], n=12)}"

def bind_evidence(item, subquery_id):
    out = dict(item)
    out["document_id"] = DOCUMENT_ID; out["source_pdf_sha256"] = BOOK_SHA256
    out["evidence_id"] = evidence_id(out); out["subquery_ids"] = [subquery_id]
    return out


def verify_evidence(item):
    errors = []
    if item.get("document_id") != DOCUMENT_ID: errors.append("document_id")
    if item.get("source_pdf_sha256") != BOOK_SHA256: errors.append("source_sha256")
    modality, source_id, page = item.get("modality"), item.get("source_object_id"), item.get("pdf_page_number")
    registry = {"text": TEXT_BY_CHUNK, "math": MATH_CARD_BY_OBJECT, "visual": VISUAL_BY_ID, "table": TABLE_BY_ID}.get(modality, {})
    source = registry.get(source_id)
    if source is None: errors.append("source_object_id"); return errors
    expected_page = source.get("page_start", source.get("pdf_page_number"))
    if modality == "text": expected_page = int(expected_page) + 1
    if expected_page is not None and int(page) != int(expected_page): errors.append("pdf_page_number")
    if modality == "visual":
        crop = Path(source["crop_path"])
        if not crop.is_file() or sha256_file(crop) != source.get("crop_sha256"): errors.append("visual_crop")
    return errors

def retrieve_lane(modality, query, subquery_id, page_constraint=None):
    if modality == "text": items = retrieve_text(query)
    elif modality == "math": items = retrieve_math(query)
    elif modality == "visual": items = retrieve_visual(query)
    elif modality == "table": items = retrieve_table(query, page_constraint=page_constraint)
    else: items = []
    if page_constraint:
        items = [x for x in items if x["pdf_page_number"] == page_constraint]
    bound = []
    for rank, item in enumerate(items, 1):
        item = bind_evidence(item, subquery_id)
        item["retrieval_metadata"] = {"within_modality_rank": rank}
        errors = verify_evidence(item)
        if errors: raise ValueError(f"Invalid {modality} evidence: {errors}")
        bound.append(item)
    return bound

def dedupe_evidence(items):
    merged = {}
    for item in items:
        key = (item["modality"], item["source_object_id"], item["pdf_page_number"])
        if key not in merged: merged[key] = item
        else: merged[key]["subquery_ids"] = list(dict.fromkeys(merged[key]["subquery_ids"] + item["subquery_ids"]))
    return list(merged.values())

def retrieve(query):
    plan = plan_query(query)
    if plan["status"] != "plan_ready":
        return {"query": query, "status": plan["status"], "requested_modalities": plan.get("modalities", []),
                "retrieved_modalities": [], "subquery_results": [], "evidence": [], "engine_calls": 0}

    collected, subquery_results, calls = [], [], 0
    for sq in plan["subqueries"]:
        local_refs = parse_references(sq["query"])
        page_constraint = local_refs["pdf_pages"][0] if len(local_refs["pdf_pages"]) == 1 else (plan["references"]["pdf_pages"][0] if len(plan["references"]["pdf_pages"]) == 1 and len(plan["subqueries"]) == 1 else None)
        for modality in sq["modalities"]:
            calls += 1
            try:
                items = retrieve_lane(modality, sq["query"], sq["subquery_id"], page_constraint)
                collected.extend(items)
                subquery_results.append({"subquery_id": sq["subquery_id"], "modality": modality,
                                         "status": "retrieved" if items else "no_evidence", "evidence_count": len(items)})
            except Exception as exc:
                subquery_results.append({"subquery_id": sq["subquery_id"], "modality": modality,
                                         "status": "retrieval_error", "evidence_count": 0, "reason": f"{type(exc).__name__}: {exc}"})

    evidence = dedupe_evidence(collected)
    evidence.sort(key=lambda x: (min(int(s.split("_")[-1]) for s in x["subquery_ids"]), MODALITY_ORDER[x["modality"]], x["evidence_id"]))
    retrieved = [m for m in MODALITY_ORDER if any(e["modality"] == m for e in evidence)]
    statuses = [x["status"] for x in subquery_results]
    status = "retrieval_failed" if statuses and all(s == "retrieval_error" for s in statuses) else "provisional_evidence" if evidence and all(s == "retrieved" for s in statuses) else "partial_provisional_evidence" if evidence else "no_evidence"
    return {"query": query, "status": status, "requested_modalities": plan["modalities"], "retrieved_modalities": retrieved,
            "subquery_results": subquery_results, "evidence": evidence, "engine_calls": calls}

# 10. Demonstration

A small set of representative queries demonstrates each retrieval lane and the unified interface. These are examples, not an independent benchmark.

In [35]:
DEMO_QUERIES = [
    "Why does dropout work less effectively in convolutional layers?",
    "Which numbered equation defines the standalone softmax function?",
    "Find the figure showing the transformer architecture.",
    "Find the table comparing GANs and VAEs.",
    "Find the softmax equation and explain what its output represents.",
    "Show the table on PDF page 286",
]

for query in DEMO_QUERIES:
    result = retrieve(query)
    print("\n" + query)
    print("status:", result["status"], "| modalities:", result["retrieved_modalities"], "| evidence:", len(result["evidence"]))
    for item in result["evidence"][:3]:
        label = item["content"].get("label") or item["content"].get("equation_label") or item["source_object_id"]
        print(" -", item["modality"], "| page", item["pdf_page_number"], "|", label)

Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]


Why does dropout work less effectively in convolutional layers?
status: provisional_evidence | modalities: ['text'] | evidence: 6
 - text | page 184 | book_f8237d393163:text:7011f7cf18
 - text | page 197 | book_f8237d393163:text:df392fc4b9
 - text | page 193 | book_f8237d393163:text:5ba7f1852a


Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]


Which numbered equation defines the standalone softmax function?
status: provisional_evidence | modalities: ['math'] | evidence: 3
 - math | page 224 | equation:12.5
 - math | page 82 | equation:5.22
 - math | page 253 | equation:12.19


...

Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]


Find the softmax equation and explain what its output represents.
status: provisional_evidence | modalities: ['text', 'math'] | evidence: 9
 - math | page 82 | equation:5.23
 - math | page 82 | equation:5.22
 - math | page 253 | equation:12.19


Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]


Show the table on PDF page 286
status: provisional_evidence | modalities: ['table'] | evidence: 1
 - table | page 286 | book_f8237d393163:p0285:table:0002:4eaebf96


# 11. Artifact Export and Final Manifest

All persistent document-specific outputs are already written inside one book directory. The final manifest records the exact source, models, counts, and SHA-256 hashes needed to identify this build.

In [36]:
def collect_artifacts(root, exclude={"manifest.json", "manifest.sha256"}):
    rows = []
    for path in sorted(Path(root).rglob("*")):
        rel = str(path.relative_to(root)).replace("\\", "/")
        if path.is_file() and rel not in exclude:
            rows.append({"path": rel, "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
    return rows

MANIFEST = {
    "schema": "ragscholar.demo_final_manifest_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "document_id": DOCUMENT_ID,
    "source": SOURCE,
    "models": {"embedding": EMBED_MODEL, "reranker": RERANK_MODEL, "siglip": SIGLIP_MODEL},
    "counts": {
        "pages": len(PAGES), "objects": len(OBJECTS), "text_chunks": len(TEXT_CHUNKS), "text_records": len(TEXT_RECORDS),
        "equations": len(EQUATION_CARDS), "visuals": len(VISUAL_RECORDS), "table_candidates": len(TABLE_CANDIDATES),
        "tables": len(TABLE_RECORDS), "table_rows": len(TABLE_ROW_RECORDS), "table_cells": len(TABLE_CELL_RECORDS),
    },
    "environment": {
        "python": sys.version.split()[0], "torch": torch.__version__, "pymupdf": package_version("PyMuPDF"),
        "faiss": package_version("faiss-cpu") or package_version("faiss-gpu"), "bm25s": package_version("bm25s"),
        "sentence_transformers": package_version("sentence-transformers"), "transformers": package_version("transformers"),
    },
    "retrieval": {"modalities": ["text", "math", "visual", "table"], "public_api": "retrieve", "answer_generation": False, "numeric_engine": False},
}
write_json(BOOK_DIR / "manifest.json", MANIFEST)
MANIFEST["artifacts"] = collect_artifacts(BOOK_DIR)
write_json(BOOK_DIR / "manifest.json", MANIFEST)

print("Build complete")
print("Artifacts:", BOOK_DIR)
print("Manifest :", BOOK_DIR / "manifest.json")

Build complete
Artifacts: C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\artifacts\books\book_f8237d393163
Manifest : C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\artifacts\books\book_f8237d393163\manifest.json


# 12. Saved-Artifact Loading Contract

The first execution is a full fresh build. The loader below restores the persisted registries, FAISS indexes, source graph, labels, and SigLIP view arrays without parsing or rebuilding document embeddings. BM25 and model runtime objects are intentionally initialized by the notebook's retrieval setup cells when starting a new kernel.

In [37]:
def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

def load_saved_build(book_dir):
    book_dir = Path(book_dir)
    manifest = json.loads((book_dir / "manifest.json").read_text(encoding="utf-8"))
    return {
        "manifest": manifest,
        "manifest_sha256": sha256_file(book_dir / "manifest.json"),
        "document": json.loads((book_dir / "canonical" / "document.json").read_text(encoding="utf-8")),
        "labels": json.loads((book_dir / "canonical" / "labels.json").read_text(encoding="utf-8")),
        "text_records": load_jsonl(book_dir / "text" / "records.jsonl"),
        "equations": load_jsonl(book_dir / "math" / "equations.jsonl"),
        "equation_cards": load_jsonl(book_dir / "math" / "equation_cards.jsonl"),
        "math_source_graph": load_jsonl(book_dir / "math" / "source_graph.jsonl"),
        "visual_records": load_jsonl(book_dir / "visual" / "records.jsonl"),
        "visual_views": load_jsonl(book_dir / "visual" / "siglip_multiview_meta.jsonl"),
        "visual_view_vectors": np.load(book_dir / "visual" / "siglip_multiview.npy"),
        "table_candidates": load_jsonl(book_dir / "table" / "candidates.jsonl"),
        "tables": load_jsonl(book_dir / "table" / "tables.jsonl"),
        "table_rows": load_jsonl(book_dir / "table" / "rows.jsonl"),
        "table_cells": load_jsonl(book_dir / "table" / "cells.jsonl"),
        "text_faiss": faiss.read_index(str(book_dir / "text" / "dense.faiss")),
        "math_faiss": faiss.read_index(str(book_dir / "math" / "dense.faiss")),
        "visual_qwen_faiss": faiss.read_index(str(book_dir / "visual" / "qwen.faiss")),
        "table_faiss": faiss.read_index(str(book_dir / "table" / "tables.faiss")) if (book_dir / "table" / "tables.faiss").is_file() else None,
        "row_faiss": faiss.read_index(str(book_dir / "table" / "rows.faiss")) if (book_dir / "table" / "rows.faiss").is_file() else None,
        "bm25_dirs": {
            "text": book_dir / "text" / "bm25",
            "math": book_dir / "math" / "bm25",
            "visual": book_dir / "visual" / "bm25",
            "visual_ocr": book_dir / "visual" / "ocr_bm25",
            "tables": book_dir / "table" / "tables_bm25",
            "rows": book_dir / "table" / "rows_bm25",
        },
    }

print("Saved-artifact loader ready:", BOOK_DIR)


Saved-artifact loader ready: C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\artifacts\books\book_f8237d393163


---
## End of Demo Final Notebook

This notebook is intentionally the compact, reproducible replacement for the experimental development notebook. It contains only the final build path and final retrieval APIs. Evaluation notebooks, manual gold creation, checkpoint repair, restoration patches, answer generation experiments, and old runtime snapshots are not part of this file.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.app import start_app

WEB_SERVER = start_app(
    retrieve_fn=retrieve,
    book_dir=BOOK_DIR,
)

RAGScholar UI: http://127.0.0.1:5000


 * Serving Flask app 'app.app'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Oct/2026 14:03:12] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 14:03:12] "GET /static/style.css HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 14:03:12] "GET /static/app.js HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 14:03:12] "GET /api/health HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 14:03:12] "GET /favicon.ico HTTP/1.1" 404 -


Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

127.0.0.1 - - [05/Oct/2026 14:03:21] "POST /api/chat HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 14:03:31] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 14:03:31] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [05/Oct/2026 14:03:31] "GET /static/app.js HTTP/1.1" 304 -
127.0.0.1 - - [05/Oct/2026 14:03:31] "GET /api/health HTTP/1.1" 200 -


...

Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

127.0.0.1 - - [05/Oct/2026 14:05:05] "POST /api/chat HTTP/1.1" 200 -


Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

127.0.0.1 - - [05/Oct/2026 14:06:13] "POST /api/chat HTTP/1.1" 200 -
